# 03 · LoRA-DPO persona (Task 5) + all generations (Tasks 2 & 6) + blind pair files
**Settings:** GPU T4 x2 · Internet On · Save & Run All (Commit).

**Needs:** outputs of notebooks **01** and **02** attached.
**Produces:** `outputs/persona_lora/adapter`, `outputs/gen/*.json`, `outputs/eval/*_pairs.jsonl` (~1–1.5 h).

In [ ]:
# Code repo (public, or add a GitHub token as a Kaggle secret if private)
REPO_URL = "https://github.com/NifraWahaj/cs5323_PA2.git"
BRANCH = "main"

import os, subprocess, sys
REPO = "/kaggle/working/sprout"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# Kaggle's PyTorch already matches the GPU runtime: do NOT reinstall torch.
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps

In [ ]:
!python -m pip install -q "trl>=0.12,<0.25" "peft>=0.13" matplotlib

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("GPUs visible:", torch.cuda.device_count())
assert torch.cuda.device_count() == 2, "Set Accelerator to GPU T4 x2"

import trl, peft; print('trl', trl.__version__, '| peft', peft.__version__)

In [ ]:
from sproutlm.kaggle_paths import find_run
SFT = find_run("sft"); DPO_HH = find_run("dpo_hh"); DPO_UF = find_run("dpo_uf")
BASE = "Qwen/Qwen2.5-0.5B"; SMOL = "HuggingFaceTB/SmolLM2-360M-Instruct"
G, E = "/kaggle/working/outputs/gen", "/kaggle/working/outputs/eval"
print(SFT, DPO_HH, DPO_UF, sep="\n")

## Task 5 · TRL DPOTrainer + LoRA on the Ramsay pairs (~10 min)

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python -m sproutlm.persona_lora --config configs/persona_lora.yaml

In [ ]:
LORA = '/kaggle/working/outputs/persona_lora/adapter'
!cat /kaggle/working/outputs/persona_lora/final_eval.json

## Sanity check on 3 prompts per model before the full runs (checkpoint path, template, EOS, length)

In [ ]:
!python -m sproutlm.generate --model {BASE}   --eval_set alpaca  --generator base    --limit 3
!python -m sproutlm.generate --model {SFT}    --eval_set alpaca  --generator sft     --limit 3
!python -m sproutlm.generate --model {DPO_UF} --eval_set alpaca  --generator dpo_uf  --limit 3
!python -m sproutlm.generate --model {DPO_HH} --eval_set hh      --generator dpo_hh  --limit 3
!python -m sproutlm.generate --model {SMOL} --adapter {LORA} --style chat_template --eval_set persona --generator smol_lora --limit 3

## Full generation: two queues in parallel, one per GPU
Greedy, `max_new_tokens=384`, `repetition_penalty=1.1` for **every** model (see `sproutlm/generate.py: DECODE`).
Each file is written as soon as its model finishes; existing files are skipped, so a rerun resumes.

In [ ]:
q0 = f"""
python -m sproutlm.generate --device cuda:0 --model {BASE}   --eval_set alpaca --generator base   --out {G}/alpaca__base.json
python -m sproutlm.generate --device cuda:0 --model {SFT}    --eval_set alpaca --generator sft    --out {G}/alpaca__sft.json
python -m sproutlm.generate --device cuda:0 --model {DPO_UF} --eval_set alpaca --generator dpo_uf --out {G}/alpaca__dpo_uf.json
"""
q1 = f"""
python -m sproutlm.generate --device cuda:1 --model {SFT}    --eval_set hh --generator sft    --out {G}/hh__sft.json
python -m sproutlm.generate --device cuda:1 --model {DPO_HH} --eval_set hh --generator dpo_hh --out {G}/hh__dpo_hh.json
python -m sproutlm.generate --device cuda:1 --model {BASE}   --eval_set hh --generator base   --out {G}/hh__base.json
python -m sproutlm.generate --device cuda:1 --model {SMOL} --style chat_template --eval_set persona --generator smol_base --out {G}/persona__smol_base.json
python -m sproutlm.generate --device cuda:1 --model {SMOL} --adapter {LORA} --style chat_template --eval_set persona --generator smol_lora --out {G}/persona__smol_lora.json
"""
import subprocess, os
os.makedirs(G, exist_ok=True)
procs = [subprocess.Popen(["bash", "-c", "set -e;" + q.strip().replace("\n", " && ")],
                          stdout=open(f"{G}/queue{i}.log", "w"), stderr=subprocess.STDOUT)
         for i, q in enumerate([q0, q1])]
codes = [p.wait() for p in procs]
print("exit codes:", codes)
!tail -n 3 {G}/queue0.log {G}/queue1.log
assert codes == [0, 0], "a generation queue failed; see the logs above"

## Blind pairwise files (A/B order randomised per prompt; model identity kept in hidden metadata)

In [ ]:
!python -m sproutlm.pairs --candidate {G}/alpaca__dpo_uf.json   --baseline {G}/alpaca__sft.json      --rubric general         --out {E}/dpo_uf_vs_sft_pairs.jsonl
!python -m sproutlm.pairs --candidate {G}/hh__dpo_hh.json       --baseline {G}/hh__sft.json          --rubric safety          --out {E}/dpo_hh_vs_sft_pairs.jsonl
!python -m sproutlm.pairs --candidate {G}/persona__smol_lora.json --baseline {G}/persona__smol_base.json --rubric persona_style   --out {E}/lora_vs_base_style_pairs.jsonl
!python -m sproutlm.pairs --candidate {G}/persona__smol_lora.json --baseline {G}/persona__smol_base.json --rubric persona_quality --out {E}/lora_vs_base_quality_pairs.jsonl
!python -m sproutlm.pairs --candidate {G}/alpaca__sft.json      --baseline {G}/alpaca__base.json     --rubric general         --out {E}/extra_sft_vs_base_pairs.jsonl

In [ ]:
# Ship the judge code with the outputs so notebook 04 works even without Internet.
!mkdir -p /kaggle/working/outputs/code && cp -r sproutlm scripts /kaggle/working/outputs/code/
import json
for line in open(f"{E}/dpo_uf_vs_sft_pairs.jsonl").readlines()[:1]:
    print(json.dumps(json.loads(line), indent=1)[:1500])

In [ ]:
# Archive this stage (easier to download/attach), then: Save Version -> "Save & Run All (Commit)".
!cd /kaggle/working && zip -qr generation_outputs.zip outputs/gen outputs/eval outputs/persona_lora/adapter outputs/persona_lora/*.json* && ls -lh /kaggle/working && du -sh /kaggle/working/outputs/*